# RNN Implemented from Scratch with Basic Layers

This notebook implements **SimpleRNN**, **LSTM**, and **GRU** cells manually using only
`tf.keras.layers.Layer` and basic `Dense` / matrix operations — no `keras.layers.SimpleRNN`,
`LSTM`, or `GRU` is used anywhere.

The task is identical to `RNN.ipynb`: predict the next value of a noisy sine wave.

### What each custom cell computes

| Cell | Equations |
|------|-----------|
| **SimpleRNN** | `h_t = tanh(W_x · x_t + W_h · h_{t-1} + b)` |
| **LSTM** | `f,i,o = σ(...)`, `g = tanh(...)`, `c_t = f⊙c_{t-1} + i⊙g`, `h_t = o⊙tanh(c_t)` |
| **GRU** | `z,r = σ(...)`, `h̃ = tanh(...)`, `h_t = (1-z)⊙h_{t-1} + z⊙h̃` |

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras

print('TensorFlow version:', tf.__version__)

## 1. Generate Time Series Data

In [ ]:
N_SAMPLES  = 1200
N_STEPS    = 30
TRAIN_SIZE = 1000

np.random.seed(42)
time   = np.linspace(0, 8 * np.pi, N_SAMPLES)
series = np.sin(time) + 0.1 * np.random.randn(N_SAMPLES)
series = series.astype(np.float32)

def make_windows(series, n_steps):
    X, y = [], []
    for i in range(len(series) - n_steps):
        X.append(series[i : i + n_steps])
        y.append(series[i + n_steps])
    return np.array(X)[..., np.newaxis], np.array(y)

X_all, y_all = make_windows(series, N_STEPS)
X_train, y_train = X_all[:TRAIN_SIZE], y_all[:TRAIN_SIZE]
X_test,  y_test  = X_all[TRAIN_SIZE:], y_all[TRAIN_SIZE:]

print('Train:', X_train.shape, y_train.shape)
print('Test: ', X_test.shape,  y_test.shape)

## 2. Custom SimpleRNN Cell

Formula: `h_t = tanh(x_t @ W_x + h_{t-1} @ W_h + b)`

In [ ]:
class SimpleRNNCell(keras.layers.Layer):
    """A single RNN time step implemented with raw weight matrices."""

    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units     = units
        self.state_size = units   # required by keras RNN wrapper

    def build(self, input_shape):
        n_inputs = input_shape[-1]
        # Input-to-hidden weights
        self.W_x = self.add_weight(
            shape=(n_inputs, self.units), initializer='glorot_uniform', name='W_x')
        # Hidden-to-hidden (recurrent) weights
        self.W_h = self.add_weight(
            shape=(self.units, self.units), initializer='orthogonal', name='W_h')
        self.b   = self.add_weight(
            shape=(self.units,), initializer='zeros', name='b')
        super().build(input_shape)

    def call(self, x_t, states):
        h_prev = states[0]                            # (batch, units)
        h_t    = tf.tanh(x_t @ self.W_x + h_prev @ self.W_h + self.b)
        return h_t, [h_t]                             # output, new_states


class CustomSimpleRNN(keras.layers.Layer):
    """Unrolls SimpleRNNCell over the full time axis and returns the last hidden state."""

    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.rnn = keras.layers.RNN(SimpleRNNCell(units), return_sequences=False)

    def call(self, x):
        return self.rnn(x)   # (batch, units)


def build_custom_rnn(n_steps, n_features=1, units=64):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_steps, n_features)),
        CustomSimpleRNN(units),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1),
    ], name='CustomSimpleRNN')
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


rnn_model = build_custom_rnn(N_STEPS)
rnn_model.summary()

## 3. Custom LSTM Cell

Equations:
```
f_t = σ(x_t @ W_xf + h_{t-1} @ W_hf + b_f)   # forget gate
i_t = σ(x_t @ W_xi + h_{t-1} @ W_hi + b_i)   # input gate
o_t = σ(x_t @ W_xo + h_{t-1} @ W_ho + b_o)   # output gate
g_t = tanh(x_t @ W_xg + h_{t-1} @ W_hg + b_g) # candidate cell
c_t = f_t ⊙ c_{t-1} + i_t ⊙ g_t
h_t = o_t ⊙ tanh(c_t)
```

In [ ]:
class LSTMCell(keras.layers.Layer):
    """LSTM cell implemented from scratch with individual gate weight matrices."""

    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units      = units
        self.state_size = [units, units]   # [h, c]

    def build(self, input_shape):
        n_inputs = input_shape[-1]
        # Pack all four gates into two large matrices for efficiency,
        # but keep them separate here for clarity.
        def _gate_weights(name):
            Wx = self.add_weight((n_inputs, self.units), initializer='glorot_uniform', name=f'W_x{name}')
            Wh = self.add_weight((self.units, self.units), initializer='orthogonal',   name=f'W_h{name}')
            b  = self.add_weight((self.units,),            initializer='zeros',        name=f'b_{name}')
            return Wx, Wh, b

        self.W_xf, self.W_hf, self.b_f = _gate_weights('f')  # forget
        self.W_xi, self.W_hi, self.b_i = _gate_weights('i')  # input
        self.W_xo, self.W_ho, self.b_o = _gate_weights('o')  # output
        self.W_xg, self.W_hg, self.b_g = _gate_weights('g')  # cell candidate
        super().build(input_shape)

    def call(self, x_t, states):
        h_prev, c_prev = states

        f_t = tf.sigmoid(x_t @ self.W_xf + h_prev @ self.W_hf + self.b_f)
        i_t = tf.sigmoid(x_t @ self.W_xi + h_prev @ self.W_hi + self.b_i)
        o_t = tf.sigmoid(x_t @ self.W_xo + h_prev @ self.W_ho + self.b_o)
        g_t = tf.tanh   (x_t @ self.W_xg + h_prev @ self.W_hg + self.b_g)

        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * tf.tanh(c_t)

        return h_t, [h_t, c_t]


class CustomLSTM(keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.rnn = keras.layers.RNN(LSTMCell(units), return_sequences=False)

    def call(self, x):
        return self.rnn(x)


def build_custom_lstm(n_steps, n_features=1, units=64):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_steps, n_features)),
        CustomLSTM(units),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1),
    ], name='CustomLSTM')
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


lstm_model = build_custom_lstm(N_STEPS)
lstm_model.summary()

## 4. Custom GRU Cell

Equations:
```
z_t = σ(x_t @ W_xz + h_{t-1} @ W_hz + b_z)          # update gate
r_t = σ(x_t @ W_xr + h_{t-1} @ W_hr + b_r)          # reset gate
h̃_t = tanh(x_t @ W_xh + (r_t ⊙ h_{t-1}) @ W_hh + b_h)  # candidate
h_t = (1 - z_t) ⊙ h_{t-1} + z_t ⊙ h̃_t
```

In [ ]:
class GRUCell(keras.layers.Layer):
    """GRU cell implemented from scratch."""

    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units      = units
        self.state_size = units

    def build(self, input_shape):
        n_inputs = input_shape[-1]

        def _gate_weights(name):
            Wx = self.add_weight((n_inputs, self.units), initializer='glorot_uniform', name=f'W_x{name}')
            Wh = self.add_weight((self.units, self.units), initializer='orthogonal',   name=f'W_h{name}')
            b  = self.add_weight((self.units,),            initializer='zeros',        name=f'b_{name}')
            return Wx, Wh, b

        self.W_xz, self.W_hz, self.b_z = _gate_weights('z')  # update gate
        self.W_xr, self.W_hr, self.b_r = _gate_weights('r')  # reset gate
        self.W_xh, self.W_hh, self.b_h = _gate_weights('h')  # candidate
        super().build(input_shape)

    def call(self, x_t, states):
        h_prev = states[0]

        z_t  = tf.sigmoid(x_t @ self.W_xz + h_prev @ self.W_hz + self.b_z)
        r_t  = tf.sigmoid(x_t @ self.W_xr + h_prev @ self.W_hr + self.b_r)
        h_tilde = tf.tanh(x_t @ self.W_xh + (r_t * h_prev) @ self.W_hh + self.b_h)

        h_t = (1.0 - z_t) * h_prev + z_t * h_tilde
        return h_t, [h_t]


class CustomGRU(keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.rnn = keras.layers.RNN(GRUCell(units), return_sequences=False)

    def call(self, x):
        return self.rnn(x)


def build_custom_gru(n_steps, n_features=1, units=64):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_steps, n_features)),
        CustomGRU(units),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1),
    ], name='CustomGRU')
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


gru_model = build_custom_gru(N_STEPS)
gru_model.summary()

## 5. Train All Three Models

In [ ]:
EPOCHS     = 50
BATCH_SIZE = 32

es_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

def train(model, name):
    print(f'\n--- Training {name} ---')
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        callbacks=[es_cb],
        verbose=1,
    )
    return history

rnn_hist  = train(rnn_model,  'CustomSimpleRNN')
lstm_hist = train(lstm_model, 'CustomLSTM')
gru_hist  = train(gru_model,  'CustomGRU')

## 6. Training Loss Comparison

In [ ]:
histories = [rnn_hist,  lstm_hist,  gru_hist]
labels    = ['CustomSimpleRNN', 'CustomLSTM', 'CustomGRU']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for h, lbl in zip(histories, labels):
    axes[0].plot(h.history['loss'],     label=f'{lbl} train')
    axes[0].plot(h.history['val_loss'], label=f'{lbl} val', linestyle='--')
    axes[1].plot(h.history['mae'],      label=f'{lbl} train')
    axes[1].plot(h.history['val_mae'],  label=f'{lbl} val',  linestyle='--')

for ax, title in zip(axes, ['MSE Loss', 'MAE']):
    ax.set_title(title); ax.legend(fontsize=7); ax.set_xlabel('Epoch')
plt.tight_layout()
plt.show()

## 7. Predictions on Test Set

In [ ]:
rnn_pred  = rnn_model.predict(X_test).flatten()
lstm_pred = lstm_model.predict(X_test).flatten()
gru_pred  = gru_model.predict(X_test).flatten()

plt.figure(figsize=(14, 4))
plt.plot(y_test,    label='Ground truth', linewidth=1.5)
plt.plot(rnn_pred,  label='CustomRNN',    alpha=0.7)
plt.plot(lstm_pred, label='CustomLSTM',   alpha=0.7)
plt.plot(gru_pred,  label='CustomGRU',    alpha=0.7)
plt.title('Test Set Predictions — Custom RNN Cells')
plt.legend()
plt.show()

## 8. Evaluation Metrics

In [ ]:
models = {
    'CustomSimpleRNN': rnn_model,
    'CustomLSTM':      lstm_model,
    'CustomGRU':       gru_model,
}

print(f"{'Model':<18}  {'Test MSE':>10}  {'Test MAE':>10}")
print('-' * 44)
for name, model in models.items():
    mse, mae = model.evaluate(X_test, y_test, verbose=0)
    print(f'{name:<18}  {mse:>10.5f}  {mae:>10.5f}')

## 9. Verify: Gate Weights Are Trainable

In [ ]:
# Inspect the custom LSTM cell weights to confirm they were updated during training
lstm_cell = lstm_model.layers[0].rnn.cell   # LSTMCell instance

print('Trainable weights in custom LSTMCell:')
for w in lstm_cell.trainable_weights:
    print(f'  {w.name:30s}  shape={w.shape}  mean={float(tf.reduce_mean(w)):.4f}')

## Summary

| Cell | Key idea | Parameters vs Keras |
|------|----------|---------------------|
| **CustomSimpleRNN** | `h_t = tanh(W_x·x + W_h·h + b)` | Same count as `keras.layers.SimpleRNN` |
| **CustomLSTM** | 4 gates, separate weight matrices | Same count as `keras.layers.LSTM` |
| **CustomGRU** | 3 gates (z, r, candidate) | Same count as `keras.layers.GRU` |

All three cells are registered as `keras.layers.Layer` subclasses, so they benefit from
automatic gradient computation, `model.save()`, and `model.summary()`.
The `keras.layers.RNN` wrapper handles the time-step loop, passing `(x_t, states)` into
each cell's `call()` method.